# Modelo 6 — Árvore de Decisão para Predição de Falhas em PSDs

Este notebook tem como objetivo desenvolver um modelo de árvore de decisão capaz de estimar se uma posição individual do sistema de Portas de Plataforma (PSD) apresentará uma falha corretiva emergencial nos próximos 7 dias.

Cada PSD será identificada pela combinação de estação, via e posição da porta. Por exemplo, `FAL_V2_PSD09` representa a posição 09 da via 2 da estação FAL.

Para construir as variáveis de entrada, serão utilizados os alarmes dos componentes associados a cada PSD, principalmente:

- `PDM`: componente móvel da porta;
- `MDU`: unidade de controle associada à posição da porta.

As ordens corretivas emergenciais ZM01 que identificam uma porta específica serão utilizadas para construir o target de falha.

O desenvolvimento será dividido nas seguintes etapas:

1. Importação das bibliotecas;
2. Carregamento dos dados;
3. Análise e preparação das bases;
4. Identificação das PSDs e de seus componentes;
5. Preparação das ordens corretivas ZM01;
6. Construção das variáveis preditoras;
7. Construção do target de falha nos próximos 7 dias;
8. Separação temporal dos dados;
9. Treinamento da árvore de decisão;
10. Avaliação e interpretação dos resultados.

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [15]:
# Localiza a raiz do projeto, independentemente de onde o Jupyter foi iniciado
diretorio_atual = Path.cwd().resolve()

if (diretorio_atual / "data").exists():
    raiz_projeto = diretorio_atual
elif (diretorio_atual.parent / "data").exists():
    raiz_projeto = diretorio_atual.parent
elif (diretorio_atual.parent.parent / "data").exists():
    raiz_projeto = diretorio_atual.parent.parent
else:
    raise FileNotFoundError("Não foi possível localizar a pasta data do projeto.")

caminho_alarmes = (
    raiz_projeto
    / "data"
    / "Alarmes"
    / "alarmes_trata_limpos_consolidado.csv"
)

caminho_zm01 = (
    raiz_projeto
    / "data"
    / "Corretiva Emergencial"
    / "processados"
    / "ZM01_limpa.xlsx"
)

print("Raiz do projeto:", raiz_projeto)
print("Base de alarmes encontrada:", caminho_alarmes.exists())
print("Base ZM01 encontrada:", caminho_zm01.exists())

Raiz do projeto: /home/kaian/Documentos/modulo7/g04
Base de alarmes encontrada: True
Base ZM01 encontrada: True


In [16]:
colunas_alarmes = [
    "TAG",
    "dt_alarme",
    "duracao_calculada_seg",
    "estacao_extraida",
    "plataforma_extraida",
    "porta_extraida",
    "Alarme"
]

alarmes = pd.read_csv(
    caminho_alarmes,
    usecols=colunas_alarmes,
    parse_dates=["dt_alarme"],
    low_memory=False
)

# Padroniza o nome da coluna para facilitar as próximas etapas
alarmes = alarmes.rename(
    columns={
        "Alarme": "tipo_alarme"
    }
)

print(f"Quantidade de registros: {len(alarmes):,}")
print(f"Quantidade de colunas: {alarmes.shape[1]}")

display(alarmes.head())

Quantidade de registros: 742,506
Quantidade de colunas: 7


,TAG,tipo_alarme,dt_alarme,duracao_calculada_seg,estacao_extraida,plataforma_extraida,porta_extraida
0,PSD_FAL_FAL2_PDM09_OBS,OBS,2026-05-12 08:16:52.657,1.013,FAL,FAL2,9.0
1,PSD_LUZ_LUZ1_PDM08_OBS,OBS,2026-05-12 08:15:09.913,1.004,LUZ,LUZ1,8.0
2,PSD_FAL_FAL1_PDM15_OBS,OBS,2026-05-12 08:14:06.657,1.013,FAL,FAL1,15.0
3,PSD_LUZ_LUZ1_PDM19_OBS,OBS,2026-05-12 08:09:31.930,0.987,LUZ,LUZ1,19.0
4,PSD_FRE_FRE1_PDM14_OBS,OBS,2026-05-12 08:05:05.830,9.017,FRE,FRE1,14.0


In [17]:
print("Dimensão da base:")
print(f"Linhas: {alarmes.shape[0]:,}")
print(f"Colunas: {alarmes.shape[1]}")

print("\nPeríodo dos alarmes:")
print("Data inicial:", alarmes["dt_alarme"].min())
print("Data final:", alarmes["dt_alarme"].max())

print("\nTipos das colunas:")
display(alarmes.dtypes.to_frame("tipo"))

print("\nValores ausentes:")
ausencias_alarmes = (
    alarmes.isna()
    .sum()
    .to_frame("quantidade_ausente")
)

ausencias_alarmes["percentual_ausente"] = (
    ausencias_alarmes["quantidade_ausente"]
    / len(alarmes)
    * 100
).round(2)

display(ausencias_alarmes)

Dimensão da base:
Linhas: 742,506
Colunas: 7

Período dos alarmes:
Data inicial: 2023-02-27 06:26:56.897000
Data final: 2026-05-12 08:16:52.657000

Tipos das colunas:


,tipo
TAG,str
tipo_alarme,str
dt_alarme,datetime64[us]
duracao_calculada_seg,float64
estacao_extraida,str
plataforma_extraida,str
porta_extraida,float64



Valores ausentes:


,quantidade_ausente,percentual_ausente
TAG,0,0.00
tipo_alarme,0,0.00
dt_alarme,0,0.00
duracao_calculada_seg,0,0.00
estacao_extraida,0,0.00
plataforma_extraida,0,0.00
porta_extraida,170,0.02


## 1. Identificação das PSDs e de seus componentes

Cada posição do sistema PSD será identificada pela combinação entre estação, via e número da porta. Os alarmes dos componentes `PDM` e `MDU` que possuem a mesma posição serão associados à mesma PSD.

Registros gerais do sistema que não identificam uma posição individual não serão utilizados neste modelo.

In [18]:
quantidade_inicial = len(alarmes)

# Padroniza a TAG antes da extração
tag_padronizada = (
    alarmes["TAG"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Identifica os componentes individualizáveis do sistema PSD
alarmes_psd = alarmes.copy()

alarmes_psd["componente"] = tag_padronizada.str.extract(
    r"(PDM|MDU)(?=\d)",
    expand=False
)

# Extrai a via de valores como FAL1, FAL2, BUT1 e BUT2
alarmes_psd["via"] = (
    alarmes_psd["plataforma_extraida"]
    .astype("string")
    .str.extract(r"(\d+)$", expand=False)
    .astype("Int64")
)

# Padroniza a estação
alarmes_psd["estacao"] = (
    alarmes_psd["estacao_extraida"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Converte o número da posição para inteiro
alarmes_psd["posicao_psd"] = (
    pd.to_numeric(
        alarmes_psd["porta_extraida"],
        errors="coerce"
    )
    .astype("Int64")
)

# Padroniza o tipo do alarme
alarmes_psd["tipo_alarme"] = (
    alarmes_psd["tipo_alarme"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Mantém apenas registros atribuíveis a uma PSD individual
alarmes_psd = alarmes_psd[
    alarmes_psd["componente"].isin(["PDM", "MDU"])
    & alarmes_psd["via"].isin([1, 2])
    & alarmes_psd["posicao_psd"].between(1, 24)
].copy()

# Cria o identificador único da PSD
alarmes_psd["id_psd"] = (
    alarmes_psd["estacao"]
    + "_V"
    + alarmes_psd["via"].astype("string")
    + "_PSD"
    + alarmes_psd["posicao_psd"]
        .astype("string")
        .str.zfill(2)
)

# Organiza os registros cronologicamente
alarmes_psd = (
    alarmes_psd
    .sort_values(["id_psd", "dt_alarme"])
    .reset_index(drop=True)
)

print(f"Registros originais: {quantidade_inicial:,}")
print(f"Registros atribuídos às PSDs: {len(alarmes_psd):,}")
print(
    "Registros gerais removidos:",
    f"{quantidade_inicial - len(alarmes_psd):,}"
)
print(f"Estações: {alarmes_psd['estacao'].nunique()}")
print(f"PSDs identificadas: {alarmes_psd['id_psd'].nunique()}")

print("\nRegistros por componente:")
display(
    alarmes_psd["componente"]
    .value_counts()
    .rename_axis("componente")
    .to_frame("quantidade")
)

display(
    alarmes_psd[
        [
            "TAG",
            "dt_alarme",
            "id_psd",
            "componente",
            "tipo_alarme",
            "duracao_calculada_seg"
        ]
    ].head()
)

Registros originais: 742,506
Registros atribuídos às PSDs: 742,336
Registros gerais removidos: 170
Estações: 11
PSDs identificadas: 526

Registros por componente:


,quantidade
componente,
PDM,678166
MDU,64170


,TAG,dt_alarme,id_psd,componente,tipo_alarme,duracao_calculada_seg
0,PSD_BUT_BUT1_PDM01_OBS,2024-01-21 23:12:46.257,BUT_V1_PSD01,PDM,OBS,2573380.513
1,PSD_P_BUT1_PDM01_CA,2024-01-21 23:12:46.257,BUT_V1_PSD01,PDM,CA,2573380.513
2,PSD_P_BUT1_PDM01_CSD,2024-01-21 23:12:46.257,BUT_V1_PSD01,PDM,CSD,2573380.513
3,PSD_P_BUT1_PDM01_FC,2024-01-21 23:12:46.257,BUT_V1_PSD01,PDM,FC,2573380.513
4,PSD_P_BUT1_PDM01_FO,2024-01-21 23:12:46.257,BUT_V1_PSD01,PDM,FO,2573380.513


## 2. Carregamento das ordens corretivas emergenciais ZM01

A base ZM01 contém as ordens de manutenção corretiva emergencial do sistema PSD. Ela será utilizada para identificar a ocorrência e o momento das falhas.

Nesta etapa, a base será apenas carregada e inspecionada. A associação entre cada ordem e uma PSD individual será realizada posteriormente por meio do local de instalação.

In [19]:
colunas_zm01 = [
    "linha_origem",
    "ordem",
    "descricao",
    "local_de_instalacao",
    "inicio_avaria_datetime",
    "falha_flag",
    "ordem_ausente_flag",
    "ordem_repetida_flag",
    "status_do_sistema_da_ordem",
    "sintoma",
    "objeto",
    "defeito",
    "causa"
]

zm01 = pd.read_excel(
    caminho_zm01,
    sheet_name="ZM01_limpa",
    usecols=colunas_zm01
)

# Garante que o início da avaria seja tratado como data e hora
zm01["inicio_avaria_datetime"] = pd.to_datetime(
    zm01["inicio_avaria_datetime"],
    errors="coerce"
)

print("Dimensão da ZM01:")
print(f"Linhas: {zm01.shape[0]:,}")
print(f"Colunas: {zm01.shape[1]}")

print("\nPeríodo das ordens:")
print(
    "Data inicial:",
    zm01["inicio_avaria_datetime"].min()
)
print(
    "Data final:",
    zm01["inicio_avaria_datetime"].max()
)

print("\nDistribuição da flag de falha:")
display(
    zm01["falha_flag"]
    .value_counts(dropna=False)
    .rename_axis("falha_flag")
    .to_frame("quantidade")
)

print("\nValores ausentes:")
ausencias_zm01 = (
    zm01.isna()
    .sum()
    .to_frame("quantidade_ausente")
)

ausencias_zm01["percentual_ausente"] = (
    ausencias_zm01["quantidade_ausente"]
    / len(zm01)
    * 100
).round(2)

display(ausencias_zm01)

display(zm01.head())

Dimensão da ZM01:
Linhas: 10,543
Colunas: 13

Período das ordens:
Data inicial: 2023-01-01 02:39:00
Data final: 2026-04-01 22:43:00

Distribuição da flag de falha:


,quantidade
falha_flag,
False,7150
True,3393



Valores ausentes:


,quantidade_ausente,percentual_ausente
linha_origem,0,0.00
ordem,1,0.01
descricao,0,0.00
local_de_instalacao,0,0.00
inicio_avaria_datetime,0,0.00
falha_flag,0,0.00
ordem_ausente_flag,0,0.00
ordem_repetida_flag,0,0.00
status_do_sistema_da_ordem,1,0.01
sintoma,0,0.00


,linha_origem,ordem,descricao,local_de_instalacao,inicio_avaria_datetime,falha_flag,ordem_ausente_flag,ordem_repetida_flag,status_do_sistema_da_ordem,sintoma,objeto,defeito,causa
0,2,53954586.0,PSD PL 01-02 Sem status para o CCO,SP04-6-PSD-BUT,2025-11-14 06:35:28,True,False,False,LIB CNPA CAPC ERRD MOME MATC NOLQ,LAÇO DE PDM ABERTO,NaN,NaN,NaN
1,3,54145839.0,PSD - PDM's não abriram PLT02,SP04-6-PSD-BUT,2026-02-27 07:33:00,False,False,False,ENTE CNPA CAPC NOLQ SCDM,TODAS AS PDM DA FACHADA NÃO ABREM,CABEAMENTO,CONTROLADOR PCC COMUNICACAO,ERRO DE COMUNICAÇÃO
2,4,52913343.0,PSD server s/ status das plataformas,SP04-6-PSD-BUT-P1S1,2024-04-18 06:00:00,False,False,False,ENTE CNPA CAPC NOLQ SCDM,PSD SERVER COM INDICAÇÃO DE FALHA NA PLA,RACK WCN,SENSOR DESTRAVADO AVARIADO,DESGASTE
3,5,53724958.0,BUT PLT 1 PORTAS NAO ABRIRAM,SP04-6-PSD-BUT-P1S1,2025-07-07 18:38:00,False,False,False,ENTE CNPA CAPC NOLQ SCDM,TODAS AS PDM DA FACHADA NÃO ABREM,CABO/CONECTORES,SWITCH TRAVADO,DESGASTE
4,6,53655728.0,PSD plt 1 - Falha geral de abertura,SP04-6-PSD-BUT-P1S1-PSSE1,2025-05-28 09:10:00,False,False,False,ENTE CNPA CAPC NOLQ SCDM,PDM NÃO ABRE,CABEAMENTO,BAIXA ISOLACAO,FALHA CBTC


## 3. Identificação das PSDs nas ordens ZM01

Para relacionar os alarmes às manutenções, será criado na ZM01 o mesmo identificador utilizado na base de alarmes.

Somente ordens cujo local de instalação informa estação, via e posição da porta podem ser atribuídas a uma PSD individual. Ordens gerais de estação, plataforma ou outros componentes serão preservadas na base original, mas não serão utilizadas como target individual.

A ocorrência de uma ordem ZM01 associada a uma PSD será utilizada como proxy de falha corretiva emergencial. A coluna `falha_flag` não será usada como filtro, pois sua ausência não significa necessariamente que não houve falha.

In [20]:
# Padroniza o local de instalação
local_padronizado = (
    zm01["local_de_instalacao"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Extrai estação, via e posição de locais como:
# SP04-6-PSD-BUT-PLT1-PP01PDM
identificacao_zm01 = local_padronizado.str.extract(
    (
        r"^SP04-6-PSD-"
        r"(?P<estacao>[A-Z]{3})-"
        r"PLT(?P<via>[12])-"
        r"PP(?P<posicao_psd>\d{2})PDM"
        r"(?:$|-)"
    )
)

# Junta a identificação extraída à base
zm01_identificada = pd.concat(
    [
        zm01.copy(),
        identificacao_zm01
    ],
    axis=1
)

# Mantém somente ordens atribuíveis a uma PSD individual
zm01_psd = zm01_identificada.dropna(
    subset=[
        "estacao",
        "via",
        "posicao_psd"
    ]
).copy()

# Converte via e posição para inteiros
zm01_psd["via"] = (
    pd.to_numeric(zm01_psd["via"], errors="coerce")
    .astype("Int64")
)

zm01_psd["posicao_psd"] = (
    pd.to_numeric(
        zm01_psd["posicao_psd"],
        errors="coerce"
    )
    .astype("Int64")
)

# Cria o mesmo identificador utilizado nos alarmes
zm01_psd["id_psd"] = (
    zm01_psd["estacao"]
    + "_V"
    + zm01_psd["via"].astype("string")
    + "_PSD"
    + zm01_psd["posicao_psd"]
        .astype("string")
        .str.zfill(2)
)

# Define o momento do evento corretivo
zm01_psd["dt_evento_corretivo"] = (
    zm01_psd["inicio_avaria_datetime"]
)

# Verifica se a PSD também existe na base de alarmes
ids_com_alarmes = set(
    alarmes_psd["id_psd"].unique()
)

zm01_psd["possui_historico_alarmes"] = (
    zm01_psd["id_psd"].isin(ids_com_alarmes)
)

# Organiza cronologicamente
zm01_psd = (
    zm01_psd
    .sort_values(["id_psd", "dt_evento_corretivo"])
    .reset_index(drop=True)
)

print(f"Ordens ZM01 totais: {len(zm01):,}")
print(
    "Ordens atribuídas a uma PSD individual:",
    f"{len(zm01_psd):,}"
)
print(
    "Ordens gerais ou de outros componentes:",
    f"{len(zm01) - len(zm01_psd):,}"
)
print(
    "PSDs diferentes presentes na ZM01:",
    zm01_psd["id_psd"].nunique()
)
print(
    "Ordens com PSD encontrada nos alarmes:",
    f"{zm01_psd['possui_historico_alarmes'].sum():,}"
)

print("\nDistribuição da flag de falha nas ordens individualizadas:")
display(
    zm01_psd["falha_flag"]
    .value_counts(dropna=False)
    .rename_axis("falha_flag")
    .to_frame("quantidade")
)

display(
    zm01_psd[
        [
            "ordem",
            "local_de_instalacao",
            "dt_evento_corretivo",
            "id_psd",
            "falha_flag",
            "possui_historico_alarmes"
        ]
    ].head()
)

Ordens ZM01 totais: 10,543
Ordens atribuídas a uma PSD individual: 9,260
Ordens gerais ou de outros componentes: 1,283
PSDs diferentes presentes na ZM01: 524
Ordens com PSD encontrada nos alarmes: 9,228

Distribuição da flag de falha nas ordens individualizadas:


,quantidade
falha_flag,
False,6220
True,3040


,ordem,local_de_instalacao,dt_evento_corretivo,id_psd,falha_flag,possui_historico_alarmes
0,52149458.0,SP04-6-PSD-BUT-PLT1-PP01PDM,2023-02-12 10:35:00,BUT_V1_PSD01,True,True
1,52485870.0,SP04-6-PSD-BUT-PLT1-PP01PDM,2023-08-25 16:53:00,BUT_V1_PSD01,False,True
2,52752879.0,SP04-6-PSD-BUT-PLT1-PP01PDM,2024-01-21 12:22:00,BUT_V1_PSD01,True,True
3,53017687.0,SP04-6-PSD-BUT-PLT1-PP01PDM,2024-06-17 05:45:00,BUT_V1_PSD01,True,True
4,53156524.0,SP04-6-PSD-BUT-PLT1-PP01PDM,2024-09-01 16:16:00,BUT_V1_PSD01,True,True


## 4. Definição do período observável para modelagem

As bases de alarmes e de manutenções possuem períodos de cobertura diferentes entre estações e vias. Por isso, não é correto considerar automaticamente como “sem falha” um período no qual os dados não estavam disponíveis.

Para cada estação e via, será definido:

- O início da cobertura dos alarmes;
- O fim da cobertura dos alarmes;
- Uma janela histórica mínima de 30 dias para calcular as variáveis preditoras;
- Um horizonte futuro completo de 7 dias para observar o target.

A cobertura será calculada por estação e via, e não pela data do último alarme de cada porta. Uma PSD sem alarmes recentes pode estar funcionando normalmente, enquanto outras portas da mesma via confirmam que o sistema continuava sendo monitorado.

In [21]:
HORIZONTE_PREVISAO_DIAS = 7
JANELA_HISTORICO_DIAS = 30

# Calcula o período de cobertura dos alarmes por estação e via
cobertura_via = (
    alarmes_psd
    .groupby(
        ["estacao", "via"],
        as_index=False
    )
    .agg(
        inicio_cobertura_alarmes=(
            "dt_alarme",
            "min"
        ),
        fim_cobertura_alarmes=(
            "dt_alarme",
            "max"
        ),
        quantidade_alarmes=(
            "dt_alarme",
            "size"
        ),
        quantidade_psds=(
            "id_psd",
            "nunique"
        )
    )
)

# Identifica todas as PSDs observadas nos alarmes
psds_observadas = (
    alarmes_psd[
        [
            "id_psd",
            "estacao",
            "via",
            "posicao_psd"
        ]
    ]
    .drop_duplicates()
)

# Associa cada PSD à cobertura de sua estação e via
cobertura_psd = psds_observadas.merge(
    cobertura_via,
    on=["estacao", "via"],
    how="left",
    validate="many_to_one"
)

# Última data conhecida na base de corretivas
ultima_data_zm01 = (
    zm01_psd["dt_evento_corretivo"]
    .max()
    .normalize()
)

# A data de referência precisa possuir sete dias futuros completos
ultima_data_referencia = (
    ultima_data_zm01
    - pd.Timedelta(
        days=HORIZONTE_PREVISAO_DIAS
    )
)

# Reserva 30 dias iniciais para a criação das features históricas
cobertura_psd["inicio_modelagem"] = (
    cobertura_psd["inicio_cobertura_alarmes"]
    .dt.normalize()
    + pd.Timedelta(
        days=JANELA_HISTORICO_DIAS
    )
)

# O fim da modelagem não pode ultrapassar:
# 1. o fim dos alarmes daquela via;
# 2. a última data com target futuro completo.
cobertura_psd["fim_modelagem"] = (
    cobertura_psd["fim_cobertura_alarmes"]
    .dt.normalize()
    .clip(upper=ultima_data_referencia)
)

# Mantém apenas intervalos válidos
cobertura_psd = cobertura_psd[
    cobertura_psd["inicio_modelagem"]
    <= cobertura_psd["fim_modelagem"]
].copy()

# Calcula quantos dias poderão ser usados para cada PSD
cobertura_psd["quantidade_dias_modelagem"] = (
    cobertura_psd["fim_modelagem"]
    - cobertura_psd["inicio_modelagem"]
).dt.days + 1

# Mantém somente eventos de PSDs que também existem nos alarmes
ids_psd_observadas = set(
    cobertura_psd["id_psd"].unique()
)

eventos_corretivos_compativeis = zm01_psd[
    zm01_psd["id_psd"].isin(
        ids_psd_observadas
    )
].copy()

quantidade_linhas_estimadas = (
    cobertura_psd["quantidade_dias_modelagem"]
    .sum()
)

print(
    "Horizonte de previsão:",
    HORIZONTE_PREVISAO_DIAS,
    "dias"
)
print(
    "Janela histórica:",
    JANELA_HISTORICO_DIAS,
    "dias"
)
print(
    "Última data disponível na ZM01:",
    ultima_data_zm01.date()
)
print(
    "Última data possível de referência:",
    ultima_data_referencia.date()
)
print(
    "Estações e vias monitoradas:",
    len(cobertura_via)
)
print(
    "PSDs com período válido:",
    len(cobertura_psd)
)
print(
    "Eventos corretivos com PSD compatível:",
    f"{len(eventos_corretivos_compativeis):,}"
)
print(
    "Quantidade estimada de linhas diárias:",
    f"{quantidade_linhas_estimadas:,}"
)

display(
    cobertura_via[
        [
            "estacao",
            "via",
            "inicio_cobertura_alarmes",
            "fim_cobertura_alarmes",
            "quantidade_alarmes",
            "quantidade_psds"
        ]
    ]
)

Horizonte de previsão: 7 dias
Janela histórica: 30 dias
Última data disponível na ZM01: 2026-04-01
Última data possível de referência: 2026-03-25
Estações e vias monitoradas: 22
PSDs com período válido: 526
Eventos corretivos com PSD compatível: 9,228
Quantidade estimada de linhas diárias: 404,206


,estacao,via,inicio_cobertura_alarmes,fim_cobertura_alarmes,quantidade_alarmes,quantidade_psds
0,BUT,1,2023-10-17 21:44:30.900,2026-05-12 05:52:14.727,54407,24
1,BUT,2,2023-10-18 13:31:20.830,2026-05-11 08:25:56.703,37398,24
2,FAL,1,2023-10-17 22:53:09.517,2026-05-12 08:14:06.657,12183,24
3,FAL,2,2023-10-18 04:37:39.913,2026-05-12 08:16:52.657,15040,24
4,FRA,1,2023-10-18 21:14:59.083,2026-04-14 18:15:03.857,67416,24
5,FRA,2,2023-10-18 17:15:18.943,2026-04-14 19:28:40.883,91825,24
6,FRE,1,2023-02-27 06:26:56.897,2026-05-12 08:05:05.830,560,22
7,FRE,2,2023-02-27 16:58:24.710,2026-05-11 18:23:52.690,1601,24
8,HIG,1,2023-10-18 22:23:13.977,2026-05-12 07:52:23.820,37852,24
9,HIG,2,2023-10-17 22:22:55.643,2026-05-12 05:48:17.740,54088,24


## 5. Construção da grade diária e do resumo de alarmes

O modelo utilizará uma linha para cada PSD em cada data de referência. Antes de calcular as janelas históricas, será criada uma grade diária que também representa corretamente os dias em que uma PSD não gerou alarmes.

Os alarmes serão resumidos diariamente por PSD, componente, tipo e duração. A grade começa no início da cobertura dos alarmes para que os 30 dias anteriores à primeira data de modelagem estejam disponíveis para o cálculo das features.

In [22]:
# Define o dia de ocorrência de cada alarme
alarmes_psd["data_alarme"] = (
    alarmes_psd["dt_alarme"].dt.normalize()
)

# Indicadores usados nas agregações diárias
alarmes_psd["indicador_pdm"] = (
    alarmes_psd["componente"] == "PDM"
).astype("int8")

alarmes_psd["indicador_mdu"] = (
    alarmes_psd["componente"] == "MDU"
).astype("int8")

alarmes_psd["indicador_critico"] = (
    alarmes_psd["duracao_calculada_seg"] > 30
).astype("int8")

# Resumo geral por PSD e dia
alarmes_diarios = (
    alarmes_psd
    .groupby(
        ["id_psd", "data_alarme"],
        as_index=False
    )
    .agg(
        qtd_alarmes_dia=("TAG", "size"),
        qtd_pdm_dia=("indicador_pdm", "sum"),
        qtd_mdu_dia=("indicador_mdu", "sum"),
        qtd_criticos_dia=("indicador_critico", "sum"),
        duracao_total_dia=("duracao_calculada_seg", "sum"),
        duracao_max_dia=("duracao_calculada_seg", "max"),
        ultimo_alarme_dia=("dt_alarme", "max")
    )
)

# Quantidade diária de cada tipo de alarme
contagem_tipos_dia = pd.crosstab(
    index=[
        alarmes_psd["id_psd"],
        alarmes_psd["data_alarme"]
    ],
    columns=alarmes_psd["tipo_alarme"]
).reset_index()

contagem_tipos_dia.columns = (
    ["id_psd", "data_alarme"]
    + [
        f"qtd_{str(tipo).lower()}_dia"
        for tipo in contagem_tipos_dia.columns[2:]
    ]
)

alarmes_diarios = alarmes_diarios.merge(
    contagem_tipos_dia,
    on=["id_psd", "data_alarme"],
    how="left",
    validate="one_to_one"
)

# Cria a grade completa, incluindo dias sem alarmes
cobertura_historica = cobertura_psd[
    [
        "id_psd",
        "estacao",
        "via",
        "posicao_psd",
        "inicio_cobertura_alarmes",
        "inicio_modelagem",
        "fim_modelagem"
    ]
].copy()

cobertura_historica["inicio_grade"] = (
    cobertura_historica["inicio_cobertura_alarmes"]
    .dt.normalize()
)

cobertura_historica["data_referencia"] = (
    cobertura_historica.apply(
        lambda linha: pd.date_range(
            linha["inicio_grade"],
            linha["fim_modelagem"],
            freq="D"
        ),
        axis=1
    )
)

grade_historica = (
    cobertura_historica
    .explode("data_referencia", ignore_index=True)
    .drop(columns="inicio_grade")
)

grade_historica["data_referencia"] = pd.to_datetime(
    grade_historica["data_referencia"]
)

grade_historica = (
    grade_historica
    .merge(
        alarmes_diarios,
        left_on=["id_psd", "data_referencia"],
        right_on=["id_psd", "data_alarme"],
        how="left",
        validate="one_to_one"
    )
    .drop(columns="data_alarme")
)

colunas_contagem_dia = [
    coluna
    for coluna in alarmes_diarios.columns
    if coluna.startswith("qtd_")
]

grade_historica[colunas_contagem_dia] = (
    grade_historica[colunas_contagem_dia].fillna(0)
)

grade_historica[
    ["duracao_total_dia", "duracao_max_dia"]
] = grade_historica[
    ["duracao_total_dia", "duracao_max_dia"]
].fillna(0)

grade_historica["teve_alarme_dia"] = (
    grade_historica["qtd_alarmes_dia"] > 0
).astype("int8")

grade_historica = (
    grade_historica
    .sort_values(["id_psd", "data_referencia"])
    .reset_index(drop=True)
)

print(
    "Dias com pelo menos um alarme:",
    f"{len(alarmes_diarios):,}"
)
print(
    "Linhas da grade com período histórico:",
    f"{len(grade_historica):,}"
)

display(alarmes_diarios.head())

Dias com pelo menos um alarme: 48,282
Linhas da grade com período histórico: 419,986


,id_psd,data_alarme,qtd_alarmes_dia,qtd_pdm_dia,qtd_mdu_dia,qtd_criticos_dia,duracao_total_dia,duracao_max_dia,ultimo_alarme_dia,qtd_ca_dia,qtd_csd_dia,qtd_fc_dia,qtd_fo_dia,qtd_lsd_dia,qtd_oa_dia,qtd_obs_dia,qtd_rce_dia,qtd_sts_dia
0,BUT_V1_PSD01,2024-01-21,226,226,0,224,3.032248e+08,2573380.513,2024-01-21 23:12:46.257,32,32,33,32,32,32,33,0,0
1,BUT_V1_PSD01,2024-04-04,2,2,0,0,1.000000e+01,5.000,2024-04-04 18:45:38.783,0,0,1,0,0,0,1,0,0
2,BUT_V1_PSD01,2024-04-09,1,1,0,0,5.000000e+00,5.000,2024-04-09 19:00:40.170,0,0,0,0,0,1,0,0,0
3,BUT_V1_PSD01,2024-04-13,155,155,0,155,2.773235e+08,2590840.957,2024-04-13 01:27:17.210,22,22,22,22,22,23,22,0,0
4,BUT_V1_PSD01,2024-05-23,112,85,27,110,8.631935e+07,2340080.183,2024-05-23 18:15:43.987,12,12,12,13,12,12,12,0,27


## 6. Construção das features históricas

As variáveis preditoras serão calculadas utilizando somente os alarmes ocorridos até o fim da data de referência. Serão consideradas janelas de 1, 7 e 30 dias para representar o comportamento imediato, semanal e mensal de cada PSD.

As features incluem contagens gerais, contagens por componente e tipo de alarme, duração acumulada, duração máxima, duração média, quantidade de dias com alarmes e tempo desde o último alarme.

In [23]:
JANELAS_FEATURES_DIAS = [1, 7, 30]

colunas_soma_diaria = (
    colunas_contagem_dia
    + ["duracao_total_dia", "teve_alarme_dia"]
)

nomes_features = {
    "qtd_alarmes_dia": "qtd_alarmes",
    "qtd_pdm_dia": "qtd_pdm",
    "qtd_mdu_dia": "qtd_mdu",
    "qtd_criticos_dia": "qtd_criticos",
    "duracao_total_dia": "duracao_total",
    "teve_alarme_dia": "qtd_dias_com_alarme"
}

for coluna in colunas_contagem_dia:
    if coluna not in nomes_features:
        nomes_features[coluna] = coluna.removesuffix("_dia")

features_calculadas = {}
colunas_features_numericas = []

for janela in JANELAS_FEATURES_DIAS:
    somas_rolantes = (
        grade_historica
        .groupby("id_psd", sort=False)[colunas_soma_diaria]
        .rolling(window=janela, min_periods=1)
        .sum()
        .reset_index(level=0, drop=True)
        .reindex(grade_historica.index)
    )

    for coluna in colunas_soma_diaria:
        nome_feature = (
            f"{nomes_features[coluna]}_{janela}d"
        )
        features_calculadas[nome_feature] = (
            somas_rolantes[coluna].to_numpy()
        )
        colunas_features_numericas.append(nome_feature)

    maximos_rolantes = (
        grade_historica
        .groupby("id_psd", sort=False)["duracao_max_dia"]
        .rolling(window=janela, min_periods=1)
        .max()
        .reset_index(level=0, drop=True)
        .reindex(grade_historica.index)
    )

    nome_maximo = f"duracao_max_{janela}d"
    features_calculadas[nome_maximo] = (
        maximos_rolantes.to_numpy()
    )
    colunas_features_numericas.append(nome_maximo)

    quantidade = somas_rolantes["qtd_alarmes_dia"].to_numpy()
    duracao_total = (
        somas_rolantes["duracao_total_dia"].to_numpy()
    )
    nome_media = f"duracao_media_{janela}d"

    features_calculadas[nome_media] = np.divide(
        duracao_total,
        quantidade,
        out=np.zeros_like(duracao_total, dtype="float64"),
        where=quantidade > 0
    )
    colunas_features_numericas.append(nome_media)

features_rolantes = pd.DataFrame(
    features_calculadas,
    index=grade_historica.index
)

# Calcula a recência do último alarme sem olhar datas futuras
ultimo_alarme_conhecido = (
    grade_historica
    .groupby("id_psd", sort=False)["ultimo_alarme_dia"]
    .ffill()
)

features_rolantes["horas_desde_ultimo_alarme"] = (
    (
        grade_historica["data_referencia"]
        + pd.Timedelta(days=1)
        - ultimo_alarme_conhecido
    )
    .dt.total_seconds()
    .div(3600)
    .clip(
        lower=0,
        upper=JANELA_HISTORICO_DIAS * 24
    )
    .fillna(JANELA_HISTORICO_DIAS * 24)
)

features_rolantes["sem_alarme_30d"] = (
    features_rolantes["qtd_alarmes_30d"] == 0
).astype("int8")

colunas_features_numericas.extend(
    [
        "horas_desde_ultimo_alarme",
        "sem_alarme_30d"
    ]
)

grade_historica = pd.concat(
    [grade_historica, features_rolantes],
    axis=1
)

print(
    "Features numéricas criadas:",
    len(colunas_features_numericas)
)
print(
    "Valores ausentes nas features:",
    int(
        grade_historica[
            colunas_features_numericas
        ].isna().sum().sum()
    )
)

Features numéricas criadas: 53
Valores ausentes nas features: 0


## 7. Construção do target de falha nos próximos 7 dias

O target será igual a `1` quando a mesma PSD possuir uma ordem corretiva emergencial ZM01 entre o dia seguinte à data de referência e os sete dias posteriores. Caso contrário, será igual a `0`.

A ocorrência no próprio dia da referência não será incluída no target. Dessa forma, alarmes registrados no dia da falha não serão usados para afirmar que o modelo antecipou um evento que já estava acontecendo.

In [24]:
# Remove da grade os 30 dias reservados exclusivamente ao histórico
base_modelagem = grade_historica[
    grade_historica["data_referencia"]
    >= grade_historica["inicio_modelagem"]
].copy()

# Converte as corretivas para eventos diários e evita duplicidade
eventos_corretivos_diarios = (
    eventos_corretivos_compativeis[
        ["id_psd", "dt_evento_corretivo"]
    ]
    .assign(
        data_evento=lambda dados: (
            dados["dt_evento_corretivo"].dt.normalize()
        )
    )
    [["id_psd", "data_evento"]]
    .drop_duplicates()
)

# Cada evento torna positivos os sete dias que o antecedem
referencias_positivas = pd.concat(
    [
        eventos_corretivos_diarios.assign(
            data_referencia=(
                eventos_corretivos_diarios["data_evento"]
                - pd.Timedelta(days=deslocamento)
            )
        )
        for deslocamento in range(
            1,
            HORIZONTE_PREVISAO_DIAS + 1
        )
    ],
    ignore_index=True
)[
    ["id_psd", "data_referencia"]
].drop_duplicates()

referencias_positivas["target_falha_7d"] = 1

base_modelagem = base_modelagem.merge(
    referencias_positivas,
    on=["id_psd", "data_referencia"],
    how="left",
    validate="one_to_one"
)

base_modelagem["target_falha_7d"] = (
    base_modelagem["target_falha_7d"]
    .fillna(0)
    .astype("int8")
)

# Variáveis de calendário conhecidas na data da previsão
base_modelagem["dia_semana"] = (
    base_modelagem["data_referencia"]
    .dt.dayofweek
    .astype("int8")
)

base_modelagem["fim_de_semana"] = (
    base_modelagem["dia_semana"]
    .isin([5, 6])
    .astype("int8")
)

base_modelagem["mes"] = (
    base_modelagem["data_referencia"]
    .dt.month
    .astype("int8")
)

distribuicao_target = (
    base_modelagem["target_falha_7d"]
    .value_counts()
    .sort_index()
    .rename_axis("target_falha_7d")
    .to_frame("quantidade")
)

distribuicao_target["percentual"] = (
    distribuicao_target["quantidade"]
    / len(base_modelagem)
    * 100
).round(2)

print(
    "Linhas da base de modelagem:",
    f"{len(base_modelagem):,}"
)
print(
    "Eventos corretivos em dias distintos:",
    f"{len(eventos_corretivos_diarios):,}"
)
display(distribuicao_target)

Linhas da base de modelagem: 404,206
Eventos corretivos em dias distintos: 7,907


,quantidade,percentual
target_falha_7d,,
0,371827,91.99
1,32379,8.01


## 8. Consolidação e validação da base pré-processada

A tabela final manterá os identificadores necessários para auditoria, as variáveis de contexto conhecidas na data de referência, as features históricas e o target.

A estação será codificada posteriormente dentro do pipeline de treinamento, depois da separação temporal. Isso evita ajustar o pré-processador com informações do conjunto de teste. Não será aplicada padronização de escala, pois árvores de decisão não dependem da escala numérica das features.

In [25]:
colunas_contexto = [
    "estacao",
    "via",
    "posicao_psd",
    "dia_semana",
    "fim_de_semana",
    "mes"
]

colunas_features_modelo = (
    colunas_contexto
    + colunas_features_numericas
)

dados_modelagem = base_modelagem[
    ["id_psd", "data_referencia"]
    + colunas_features_modelo
    + ["target_falha_7d"]
].copy()

# Reduz o consumo de memória sem alterar o significado das features
dados_modelagem["via"] = (
    dados_modelagem["via"].astype("int8")
)
dados_modelagem["posicao_psd"] = (
    dados_modelagem["posicao_psd"].astype("int8")
)

for coluna in colunas_features_numericas:
    if coluna.startswith("qtd_") or coluna == "sem_alarme_30d":
        dados_modelagem[coluna] = pd.to_numeric(
            dados_modelagem[coluna],
            downcast="integer"
        )
    else:
        dados_modelagem[coluna] = pd.to_numeric(
            dados_modelagem[coluna],
            downcast="float"
        )

# Validações que devem ser satisfeitas antes do treinamento
quantidade_duplicadas = dados_modelagem.duplicated(
    ["id_psd", "data_referencia"]
).sum()

quantidade_ausentes = dados_modelagem[
    colunas_features_modelo
].isna().sum().sum()

matriz_numerica = dados_modelagem[
    colunas_features_numericas
].to_numpy(dtype="float64")

quantidade_nao_finitos = (
    ~np.isfinite(matriz_numerica)
).sum()

assert quantidade_duplicadas == 0
assert quantidade_ausentes == 0
assert quantidade_nao_finitos == 0
assert dados_modelagem["target_falha_7d"].nunique() == 2

X_base = dados_modelagem[colunas_features_modelo].copy()
y = dados_modelagem["target_falha_7d"].copy()

memoria_mb = (
    dados_modelagem.memory_usage(deep=True).sum()
    / 1024 ** 2
)

print(f"Linhas finais: {len(dados_modelagem):,}")
print(f"PSDs finais: {dados_modelagem['id_psd'].nunique()}")
print(
    "Período final:",
    dados_modelagem["data_referencia"].min().date(),
    "até",
    dados_modelagem["data_referencia"].max().date()
)
print(f"Features do modelo: {len(colunas_features_modelo)}")
print(f"Linhas duplicadas: {quantidade_duplicadas}")
print(f"Valores ausentes: {quantidade_ausentes}")
print(f"Valores não finitos: {quantidade_nao_finitos}")
print(f"Memória aproximada: {memoria_mb:.1f} MB")

display(dados_modelagem.head())

Linhas finais: 404,206
PSDs finais: 526
Período final: 2023-03-29 até 2026-03-25
Features do modelo: 59
Linhas duplicadas: 0
Valores ausentes: 0
Valores não finitos: 0
Memória aproximada: 109.5 MB


,id_psd,data_referencia,estacao,via,posicao_psd,dia_semana,fim_de_semana,mes,qtd_alarmes_1d,qtd_pdm_1d,qtd_mdu_1d,qtd_criticos_1d,qtd_ca_1d,qtd_csd_1d,qtd_fc_1d,qtd_fo_1d,qtd_lsd_1d,qtd_oa_1d,qtd_obs_1d,qtd_rce_1d,qtd_sts_1d,duracao_total_1d,qtd_dias_com_alarme_1d,duracao_max_1d,duracao_media_1d,qtd_alarmes_7d,qtd_pdm_7d,qtd_mdu_7d,qtd_criticos_7d,qtd_ca_7d,qtd_csd_7d,qtd_fc_7d,qtd_fo_7d,qtd_lsd_7d,qtd_oa_7d,qtd_obs_7d,qtd_rce_7d,qtd_sts_7d,duracao_total_7d,qtd_dias_com_alarme_7d,duracao_max_7d,duracao_media_7d,qtd_alarmes_30d,qtd_pdm_30d,qtd_mdu_30d,qtd_criticos_30d,qtd_ca_30d,qtd_csd_30d,qtd_fc_30d,qtd_fo_30d,qtd_lsd_30d,qtd_oa_30d,qtd_obs_30d,qtd_rce_30d,qtd_sts_30d,duracao_total_30d,qtd_dias_com_alarme_30d,duracao_max_30d,duracao_media_30d,horas_desde_ultimo_alarme,sem_alarme_30d,target_falha_7d
0,BUT_V1_PSD01,2023-11-16,BUT,1,1,3,0,11,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0,0.0,0.0,720.0,1,0
1,BUT_V1_PSD01,2023-11-17,BUT,1,1,4,0,11,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0,0.0,0.0,720.0,1,0
2,BUT_V1_PSD01,2023-11-18,BUT,1,1,5,1,11,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0,0.0,0.0,720.0,1,0
3,BUT_V1_PSD01,2023-11-19,BUT,1,1,6,1,11,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0,0.0,0.0,720.0,1,0
4,BUT_V1_PSD01,2023-11-20,BUT,1,1,0,0,11,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0,0.0,0.0,720.0,1,0


## 9. Divisão temporal em treino, validação e teste

A base é separada cronologicamente, sem embaralhamento, para simular o uso real do modelo: ele aprende com o passado e é avaliado em períodos posteriores.

Como o target considera falhas nos 7 dias seguintes, são retirados 7 dias entre treino e validação e entre validação e teste. Essa purga impede a sobreposição das janelas futuras usadas para construir o target e reduz o risco de vazamento temporal.

- **Treino:** 29/03/2023 a 24/03/2025;
- **Validação:** 01/04/2025 a 23/09/2025;
- **Teste:** 01/10/2025 a 25/03/2026.

In [26]:
DATA_INICIO_VALIDACAO = pd.Timestamp("2025-04-01")
DATA_INICIO_TESTE = pd.Timestamp("2025-10-01")
DIAS_PURGA = HORIZONTE_PREVISAO_DIAS

# O último target de cada conjunto deve terminar antes do próximo período
DATA_FIM_TREINO = (
    DATA_INICIO_VALIDACAO
    - pd.Timedelta(days=DIAS_PURGA + 1)
)
DATA_FIM_VALIDACAO = (
    DATA_INICIO_TESTE
    - pd.Timedelta(days=DIAS_PURGA + 1)
)

mascara_treino = (
    dados_modelagem["data_referencia"] <= DATA_FIM_TREINO
)
mascara_validacao = (
    dados_modelagem["data_referencia"] >= DATA_INICIO_VALIDACAO
) & (
    dados_modelagem["data_referencia"] <= DATA_FIM_VALIDACAO
)
mascara_teste = (
    dados_modelagem["data_referencia"] >= DATA_INICIO_TESTE
)

dados_treino = dados_modelagem.loc[mascara_treino].copy()
dados_validacao = dados_modelagem.loc[mascara_validacao].copy()
dados_teste = dados_modelagem.loc[mascara_teste].copy()

mascara_utilizada = (
    mascara_treino | mascara_validacao | mascara_teste
)
dados_purga = dados_modelagem.loc[~mascara_utilizada].copy()

X_treino = dados_treino[colunas_features_modelo].copy()
y_treino = dados_treino["target_falha_7d"].copy()

X_validacao = dados_validacao[colunas_features_modelo].copy()
y_validacao = dados_validacao["target_falha_7d"].copy()

X_teste = dados_teste[colunas_features_modelo].copy()
y_teste = dados_teste["target_falha_7d"].copy()

def resumir_conjunto(nome, dados):
    return {
        "conjunto": nome,
        "data_inicial": dados["data_referencia"].min().date(),
        "data_final": dados["data_referencia"].max().date(),
        "linhas": len(dados),
        "psds": dados["id_psd"].nunique(),
        "falhas_positivas": int(dados["target_falha_7d"].sum()),
        "taxa_falha_pct": (
            dados["target_falha_7d"].mean() * 100
        )
    }

resumo_divisao_temporal = pd.DataFrame([
    resumir_conjunto("Treino", dados_treino),
    resumir_conjunto("Validação", dados_validacao),
    resumir_conjunto("Teste", dados_teste)
])
resumo_divisao_temporal["taxa_falha_pct"] = (
    resumo_divisao_temporal["taxa_falha_pct"].round(2)
)

# Validações da separação temporal
assert len(dados_treino) > 0
assert len(dados_validacao) > 0
assert len(dados_teste) > 0
assert (
    len(dados_treino)
    + len(dados_validacao)
    + len(dados_teste)
    + len(dados_purga)
    == len(dados_modelagem)
)
assert (
    dados_treino["data_referencia"].max()
    + pd.Timedelta(days=HORIZONTE_PREVISAO_DIAS)
    < dados_validacao["data_referencia"].min()
)
assert (
    dados_validacao["data_referencia"].max()
    + pd.Timedelta(days=HORIZONTE_PREVISAO_DIAS)
    < dados_teste["data_referencia"].min()
)
assert y_treino.nunique() == 2
assert y_validacao.nunique() == 2
assert y_teste.nunique() == 2
assert set(dados_validacao["id_psd"]).issubset(
    set(dados_treino["id_psd"])
)
assert set(dados_teste["id_psd"]).issubset(
    set(dados_treino["id_psd"])
)

display(resumo_divisao_temporal)
print(f"Linhas reservadas para purga temporal: {len(dados_purga):,}")

,conjunto,data_inicial,data_final,linhas,psds,falhas_positivas,taxa_falha_pct
0,Treino,2023-03-29,2025-03-24,245386,526,19054,7.76
1,Validação,2025-04-01,2025-09-23,84128,478,7311,8.69
2,Teste,2025-10-01,2026-03-25,68000,478,5417,7.97


Linhas reservadas para purga temporal: 6,692


## 10. Pré-processamento para a Árvore de Decisão

As variáveis categóricas são transformadas com *one-hot encoding*, ajustado exclusivamente no conjunto de treino. Categorias desconhecidas em períodos futuros são ignoradas pelo codificador.

As variáveis numéricas são mantidas na escala original. Árvores de decisão dividem os dados por limiares e, por isso, não precisam de `StandardScaler` ou `MinMaxScaler`. `id_psd` e `data_referencia` não entram como features do modelo.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

colunas_categoricas_modelo = [
    "estacao",
    "via",
    "dia_semana",
    "mes"
]

colunas_numericas_pipeline = [
    coluna
    for coluna in colunas_features_modelo
    if coluna not in colunas_categoricas_modelo
]

assert set(colunas_categoricas_modelo).issubset(
    colunas_features_modelo
)
assert (
    len(colunas_categoricas_modelo)
    + len(colunas_numericas_pipeline)
    == len(colunas_features_modelo)
)

preprocessador_arvore = ColumnTransformer(
    transformers=[
        (
            "categoricas",
            OneHotEncoder(handle_unknown="ignore"),
            colunas_categoricas_modelo
        ),
        (
            "numericas",
            "passthrough",
            colunas_numericas_pipeline
        )
    ],
    remainder="drop",
    sparse_threshold=1.0,
    verbose_feature_names_out=False
)

print(
    "Features categóricas:",
    len(colunas_categoricas_modelo)
)
print(
    "Features numéricas:",
    len(colunas_numericas_pipeline)
)

## 11. Treinamento da Árvore de Decisão inicial

Este primeiro modelo funciona como baseline. A profundidade e o tamanho mínimo das folhas são limitados para reduzir a memorização dos dados, enquanto `class_weight='balanced'` aumenta a atenção dada à classe minoritária de falha.

O pipeline ajusta o codificador e a árvore somente com o conjunto de treino. Os hiperparâmetros serão comparados posteriormente usando apenas a validação.

In [ ]:
arvore_decisao_base = DecisionTreeClassifier(
    criterion="gini",
    max_depth=8,
    min_samples_split=500,
    min_samples_leaf=250,
    class_weight="balanced",
    random_state=42
)

pipeline_arvore_base = Pipeline(steps=[
    ("preprocessamento", preprocessador_arvore),
    ("modelo", arvore_decisao_base)
])

pipeline_arvore_base.fit(X_treino, y_treino)

arvore_treinada = pipeline_arvore_base.named_steps["modelo"]
nomes_features_transformadas = (
    pipeline_arvore_base
    .named_steps["preprocessamento"]
    .get_feature_names_out()
)

print("Treinamento concluído.")
print(f"Features após a codificação: {len(nomes_features_transformadas)}")
print(f"Profundidade efetiva da árvore: {arvore_treinada.get_depth()}")
print(f"Quantidade de folhas: {arvore_treinada.get_n_leaves()}")

## 12. Avaliação inicial no conjunto de validação

O modelo é comparado entre treino e validação para identificar sinais de sobreajuste. Como as falhas representam menos de 10% das linhas, a análise prioriza precisão, recall, F1, PR-AUC e acurácia balanceada; a acurácia comum é apresentada apenas como referência.

O conjunto de teste ainda não é utilizado nesta etapa.

In [ ]:
def calcular_metricas_classificacao(y_real, y_previsto, y_probabilidade):
    return {
        "acuracia": accuracy_score(y_real, y_previsto),
        "acuracia_balanceada": balanced_accuracy_score(
            y_real,
            y_previsto
        ),
        "precisao": precision_score(
            y_real,
            y_previsto,
            zero_division=0
        ),
        "recall": recall_score(
            y_real,
            y_previsto,
            zero_division=0
        ),
        "f1": f1_score(
            y_real,
            y_previsto,
            zero_division=0
        ),
        "pr_auc": average_precision_score(
            y_real,
            y_probabilidade
        ),
        "roc_auc": roc_auc_score(
            y_real,
            y_probabilidade
        ),
        "taxa_alertas": y_previsto.mean()
    }

previsoes_treino = pipeline_arvore_base.predict(X_treino)
probabilidades_treino = (
    pipeline_arvore_base.predict_proba(X_treino)[:, 1]
)

previsoes_validacao = pipeline_arvore_base.predict(X_validacao)
probabilidades_validacao = (
    pipeline_arvore_base.predict_proba(X_validacao)[:, 1]
)

metricas_arvore_base = pd.DataFrame({
    "Treino": calcular_metricas_classificacao(
        y_treino,
        previsoes_treino,
        probabilidades_treino
    ),
    "Validação": calcular_metricas_classificacao(
        y_validacao,
        previsoes_validacao,
        probabilidades_validacao
    )
})

matriz_confusao_validacao = confusion_matrix(
    y_validacao,
    previsoes_validacao,
    labels=[0, 1]
)

acuracia_sem_alertas = 1 - y_validacao.mean()

display(metricas_arvore_base.round(4))
print(
    "Acurácia de prever sempre 'sem falha':",
    f"{acuracia_sem_alertas:.2%}"
)

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(
    matriz_confusao_validacao,
    annot=True,
    fmt=",d",
    cmap="Blues",
    cbar=False,
    xticklabels=["Sem falha", "Falha"],
    yticklabels=["Sem falha", "Falha"],
    ax=ax
)
ax.set_title("Matriz de confusão — validação")
ax.set_xlabel("Classe prevista")
ax.set_ylabel("Classe real")
plt.tight_layout()
plt.show()